In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})


In [ ]:

folderVec = [
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_2_75D",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_40dx_test_3_75D",
]

farNearFolder = [
    ["farfield2", "probe3"],
    ["farfield4", "probe3"],
]

lblVec = [
    "Present LBM Cumulant D/dx=25",
    "Present LBM Cumulant D/dx=40",
]

DPhy = 1.0

DLbVec = [
    25,
    40,
]

# both cases (D/dx=25 and D/dx=40) keep 696 Gaussian points = 232 triangles
# total area ≈ 232 × (8.24 m² / 9356 tri) ≈ 0.204 m²
# circumference(D=1.5) = 4.71 m → Lz_eff ≈ 0.043 m
deltaZEff = 0.043

rhoPhyVec = [
    1.204,
    1.204,
]
gamma = 1.0

U0PhyVec = [
    68.0,
    68.0,
]

stepRg = [
    [60000, 100000],
    [60000, 100000],
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]
# compareNearFar = "near"
compareNearFar = "far"
# compareNearFar = "both"

dxVec = [ DPhy/DLb for DLb in DLbVec ]
unitConverterVec = [ uc(dx, rhoPhy, gamma) for dx, rhoPhy in zip(dxVec, rhoPhyVec) ]
pres0PhyVec = [ unitConverter.lb_to_phys_pressure(1.0 * unitConverter.cs2_lb) for unitConverter in unitConverterVec ]


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

def preFreqDomain_to_splFreqDomain(preFreqDomain, refPressure):
    spl = 20 * np.log10(np.abs(preFreqDomain) / np.sqrt(2) / refPressure)
    return spl

def splFreqDomain_to_psdFreqDomain(splFreqDomain, deltaFreq):
    psd = splFreqDomain - 10 * np.log10(deltaFreq)
    return psd

def preFreqDomain_to_psdFreqDomain(preFreqDomain, refPressure, deltaFreq):
    spl = preFreqDomain_to_splFreqDomain(preFreqDomain, refPressure)
    psd = splFreqDomain_to_psdFreqDomain(spl, deltaFreq)
    return psd

def correct_factor_3d_to_2d(freq, r, cs, spanL):
    return (np.sqrt(r * cs / freq) / spanL)

def correct_3d_to_2d(p3DTimeDomain, stepVec, r, cs, deltaZ, unitConverter):
    dtSampling = unitConverter.step_to_time(stepVec[1] - stepVec[0])
    n = len(p3DTimeDomain)
    pMean = np.mean(p3DTimeDomain)
    pFluct = p3DTimeDomain - pMean
    pHat = np.fft.rfft(pFluct)
    freq = np.fft.rfftfreq(n, d=dtSampling)

    correctFactor = np.zeros(len(freq), dtype=complex)
    idx = np.arange(1, len(freq))
    correctFactor[idx] = (np.sqrt(r * cs / freq[idx]) / deltaZ) * np.exp(1j * np.pi / 4.0)
    pHat2D = pHat * correctFactor
    pFluct2D = np.fft.irfft(pHat2D, n=n)
    p2DTimeDomain = pFluct2D + pMean
    return p2DTimeDomain

In [ ]:
farRadVecVec = []
farPresRMSVecVec = []

nearRadVecVec = []
nearPresRMSVecVec = []

for case in range(0, len(folderVec)):
    if compareNearFar == "far" or compareNearFar == "both":
        print("wow far")
        farFileList = sorted(glob.glob(f"{folderVec[case]}/microphones/{farNearFolder[case][0]}/microphone*.txt"))
        farRadVec = []
        farPresFluctuateRMSVec = []
        print(len(farFileList))
        for iFar in range(0, len(farFileList)):
            x, y, z = get_probe_coords(farFileList[iFar])
            radians = np.arctan2(y, x) % (2 * np.pi)
            farRadVec.append(radians)
            stepCol = read_col_probe(farFileList[iFar], 0, 2)
            stepCol = stepCol - stepCol[0]
            pressureCol = read_col_probe(farFileList[iFar], 2, 2)
            idxStart = int(np.abs(stepCol - stepRg[case][0]).argmin())
            pressure2D = correct_3d_to_2d(pressureCol[idxStart:], stepCol[idxStart:], np.sqrt(x**2+y**2), unitConverterVec[case].cs_phy, deltaZEff, unitConverterVec[case])
            deltaPres = pressure2D # 原本已经减去了环境压力
            deltaPresTilde = deltaPres - np.mean(deltaPres)

            deltaPresTildeRMS = np.sqrt(np.mean(deltaPresTilde**2))
            # farPresFluctuateRMSVec.append(deltaPresTildeRMS) # physical unit
            # deltaPresTildeRMSLB = unitConverterVec[case].phys_to_lb_pressure(deltaPresTildeRMS) # lb unit
            deltaPresTildeRMSLB = unitConverterVec[case].phys_to_lb_pressure(deltaPresTildeRMS) / (unitConverterVec[case].cs2_lb) # normalized
            farPresFluctuateRMSVec.append(deltaPresTildeRMSLB)
        farRadVecVec.append(farRadVec)
        farPresRMSVecVec.append(farPresFluctuateRMSVec)

    if compareNearFar == "near" or compareNearFar == "both":
        print("wow near")
        nearRadVec = []
        nearPresFluctuateRMSVec = []
        nearFileList = sorted(glob.glob(f"{folderVec[case]}/probes/{farNearFolder[case][1]}/probe*.txt"))
        for iNear in range(0, len(nearFileList)):
            x, y, z = get_probe_coords(nearFileList[iNear])
            radians = np.arctan2(y, x) % (2 * np.pi)
            nearRadVec.append(radians)
            stepCol = read_col_probe(nearFileList[iNear], 0, 2)
            rhoCol = read_col_probe(nearFileList[iNear], 2, 2)
            idxStart = int(np.abs(stepCol - stepRg[case][0]).argmin())
            deltaPres = (rhoCol - 1.0) * unitConverterVec[case].cs2_lb
            deltaPresTilde = deltaPres[idxStart:] - np.mean(deltaPres[idxStart:])
            # deltaPresTildeRMS = np.sqrt(np.mean(deltaPresTilde**2)) # lb unit
            deltaPresTildeRMS = np.sqrt(np.mean(deltaPresTilde**2)) / (unitConverterVec[case].cs2_lb) # normalized
        nearRadVecVec.append(nearRadVec)
        nearPresRMSVecVec.append(nearPresFluctuateRMSVec)
    # print(farRadVecVec)
    # print(nearRadVecVec)



In [ ]:
# ref1File = "../ref/directivity_rmsPTilde_inoue_Ma02.csv"
ref1File = "../ref/directivity_rmsPTilde_modified_inoue_Ma02.csv"
ref1Angles = read_csv_col(ref1File, 1, 1)
ref1PFluct = read_csv_col(ref1File, 0, 1)
ref1Radians = np.radians(ref1Angles)

In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 8), facecolor='w', edgecolor='w', subplot_kw={'projection': 'polar'})

ax1.plot(ref1Radians, ref1PFluct, marker='o', fillstyle='none', mew=2, lw=0, c='k', label="Inoue & Hatakeyama, DNS (D/dx=200)")

for case in range(0, len(folderVec)):
    if compareNearFar == "far" or compareNearFar == "both":
        ax1.plot(farRadVecVec[case], farPresRMSVecVec[case], label=rf"{lblVec[case]} FWH (3D$\to$2D)", lw=2, c=colorVec[case])
    if compareNearFar == "near" or compareNearFar == "both":
        degArray = np.degrees(nearRadVecVec[case])
        angle = 58
        mask = ~((degArray <= angle) | (degArray >= (360-angle)))
        ax1.plot(np.array(nearRadVecVec[case])[mask], np.array(nearPresRMSVecVec[case])[mask], linestyle=':', lw=2, c=colorVec[case], label=f"{lblVec} nearfield")

ax1.set_ylim(0, 1E-4)
lines, labels = ax1.set_thetagrids(np.arange(0, 360, 30))
ax1.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25), frameon=False)

ticks = np.linspace(0, 1E-4, 5)
ax1.set_rticks(ticks)
labelOuter = r"$1.0 \times 10^{-4}$" + "\n" + r"$p\prime_{RMS}/(\rho_0c_s^2)$"
labels = [
    "0.0",
    "",
    "0.5",
    "",
    labelOuter
]
ax1.set_yticklabels(labels)
ax1.set_rlabel_position(15)
